# Unidad 6 · Correlación

Cuaderno de la base abierta de ingeniería (Análisis de Datos). Reproduce el ejemplo resuelto (el
carrito de mote con huesillo) y la pieza del caso Estampa Sur (precio contra unidades vendidas). Lee
los datos de `../datos/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

DATOS = Path("..") / "datos"
RES = {}   # resultados que el verificador compara

## Ejemplo · temperatura y vasos vendidos

Primero el gráfico, después los coeficientes. La prueba de Pearson usa $t = r\sqrt{n-2}/\sqrt{1-r^2}$
con $n-2$ grados de libertad; `pearsonr` hace la misma cuenta.

In [ ]:
mote = pd.read_csv(DATOS / "u06_mote.csv")
x, y = mote.temperatura, mote.vasos
plt.scatter(x, y); plt.xlabel("temperatura máxima (°C)"); plt.ylabel("vasos"); plt.show()

dx, dy = x - x.mean(), y - y.mean()
r_mano = (dx * dy).sum() / np.sqrt((dx**2).sum() * (dy**2).sum())
t = r_mano * np.sqrt(len(x) - 2) / np.sqrt(1 - r_mano**2)
pr = stats.pearsonr(x, y)
print(f"r a mano = {r_mano:.4f}; t = {t:.2f}; pearsonr: r = {pr.statistic:.4f}, p = {pr.pvalue:.5f}")
print("Spearman:", stats.spearmanr(x, y))
print("Kendall: ", stats.kendalltau(x, y))
RES.update(ej_r=pr.statistic, ej_p=pr.pvalue, ej_rho=stats.spearmanr(x, y).statistic,
           ej_tau=stats.kendalltau(x, y).statistic)

## Caso · Estampa Sur, precio y unidades vendidas en 30 semanas

Además de los tres coeficientes, la correlación parcial controlando la publicidad: la correlación
entre lo que queda del precio y de las unidades después de sacarles lo que explica la publicidad.

In [ ]:
v = pd.read_csv(DATOS / "ventas.csv")
plt.scatter(v.precio, v.unidades); plt.xlabel("precio ($)"); plt.ylabel("unidades"); plt.show()

pr = stats.pearsonr(v.precio, v.unidades)
sp = stats.spearmanr(v.precio, v.unidades)
kt = stats.kendalltau(v.precio, v.unidades)
cov = v[["precio", "unidades"]].cov().iloc[0, 1]
print(f"cov = {cov:.0f}; r = {pr.statistic:.3f} (p = {pr.pvalue:.6f}); IC 95 %: {pr.confidence_interval()}")
print(f"Spearman = {sp.statistic:.3f} (p = {sp.pvalue:.6f}); Kendall = {kt.statistic:.3f} (p = {kt.pvalue:.6f})")

r = v[["precio", "unidades", "publicidad"]].corr()
rxy, rxz, ryz = r.loc["precio", "unidades"], r.loc["precio", "publicidad"], r.loc["unidades", "publicidad"]
r_parcial = (rxy - rxz * ryz) / np.sqrt((1 - rxz**2) * (1 - ryz**2))
print(r.round(3)); print(f"correlación parcial precio-unidades, controlando publicidad: {r_parcial:.3f}")
RES.update(caso_r=pr.statistic, caso_p=pr.pvalue, caso_rho=sp.statistic, caso_p_rho=sp.pvalue,
           caso_tau=kt.statistic, caso_cov=cov, caso_r_parcial=r_parcial)